In [28]:
import pandas as pd

wanted_types = [
    "metadata_json",
    "narration_atomic_action_csv",
    "narration_activity_summarization_csv",
]

rows = []

for sequence_id, sequence_data in manifest["sequences"].items():
    for file_type in wanted_types:
        if file_type in sequence_data:
            entry = sequence_data[file_type]
            rows.append({
                "sequence_id": sequence_id,
                "file_type": file_type,
                "filename": entry["filename"],
                "file_size_bytes": entry["file_size_bytes"],
                "download_url": entry["download_url"],
                "sha1sum": entry.get("sha1sum")
            })

df_files = pd.DataFrame(rows)

print("Rows:", len(df_files))
print("Unique sequences:", df_files["sequence_id"].nunique())
df_files.head()

Rows: 2809
Unique sequences: 1100


,sequence_id,file_type,filename,file_size_bytes,download_url,sha1sum
0,20230607_s0_james_johnson_act0_e72nhq,metadata_json,Nymeria_v0.0_20230607_s0_james_johnson_act0_e7...,1367,https://scontent.xx.fbcdn.net/m1/v/t6/An-h6phj...,cf9a90058d0bd4df1d8c56329e33d540d585b22c
1,20230607_s0_james_johnson_act0_e72nhq,narration_atomic_action_csv,Nymeria_v0.0_20230607_s0_james_johnson_act0_e7...,46391,https://scontent.xx.fbcdn.net/m1/v/t6/An84jpu8...,bb083d6b7f21b2908939550ccb4e5110c0773509
2,20230607_s0_james_johnson_act0_e72nhq,narration_activity_summarization_csv,Nymeria_v0.0_20230607_s0_james_johnson_act0_e7...,7694,https://scontent.xx.fbcdn.net/m1/v/t6/An_HJbws...,979cfbc78b3a21533739497b93416b787a555524
3,20230607_s0_james_johnson_act1_7xwm28,metadata_json,Nymeria_v0.0_20230607_s0_james_johnson_act1_7x...,1364,https://scontent.xx.fbcdn.net/m1/v/t6/An_hhJNN...,a1798628300502dd21afbb25f075e1207da2e7c5
4,20230607_s0_james_johnson_act1_7xwm28,narration_atomic_action_csv,Nymeria_v0.0_20230607_s0_james_johnson_act1_7x...,56078,https://scontent.xx.fbcdn.net/m1/v/t6/An_nB1rG...,0a9af1cc54c52c617b5efa1c2c72d1e9fa52cca5


In [29]:
summary = (
    df_files.groupby("file_type")["file_size_bytes"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

summary["sum_mb"] = summary["sum"] / (1024 ** 2)
summary["mean_mb"] = summary["mean"] / (1024 ** 2)

summary

,file_type,count,sum,mean,sum_mb,mean_mb
0,metadata_json,1100,1513984,1376.349091,1.443848,0.001313
1,narration_activity_summarization_csv,864,5743251,6647.281250,5.477191,0.006339
2,narration_atomic_action_csv,845,52533137,62169.392899,50.099504,0.059289


In [30]:
df_all = df_files.copy()

print("Files to download:", len(df_all))
print("Sequences covered:", df_all["sequence_id"].nunique())

Files to download: 2809
Sequences covered: 1100


In [31]:
import os
import tempfile
import requests
import boto3

s3 = boto3.client("s3")
BUCKET = "sagemaker-bst"
TARGET_PREFIX = "nymeria/lightweight/"

def stream_url_to_s3(url: str, bucket: str, key: str, timeout: int = 120):
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with tempfile.NamedTemporaryFile(delete=False) as tmp:
            tmp_path = tmp.name
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    tmp.write(chunk)

    s3.upload_file(tmp_path, bucket, key)
    os.remove(tmp_path)

In [32]:
existing_keys = set()

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=TARGET_PREFIX, MaxKeys=1000)
for obj in resp.get("Contents", []):
    existing_keys.add(obj["Key"])

print("Existing objects already in bucket:", len(existing_keys))

Existing objects already in bucket: 9


In [ ]:
download_log = []

for _, row in df_all.iterrows():
    s3_key = f"{TARGET_PREFIX}{row['sequence_id']}/{row['filename']}"
    
    if s3_key in existing_keys:
        download_log.append({
            "sequence_id": row["sequence_id"],
            "file_type": row["file_type"],
            "filename": row["filename"],
            "s3_key": s3_key,
            "status": "already_exists"
        })
        continue

    try:
        stream_url_to_s3(
            url=row["download_url"],
            bucket=BUCKET,
            key=s3_key
        )
        download_log.append({
            "sequence_id": row["sequence_id"],
            "file_type": row["file_type"],
            "filename": row["filename"],
            "s3_key": s3_key,
            "status": "ok"
        })
        print("Uploaded:", s3_key)
        
    except Exception as e:
        download_log.append({
            "sequence_id": row["sequence_id"],
            "file_type": row["file_type"],
            "filename": row["filename"],
            "s3_key": s3_key,
            "status": f"error: {str(e)}"
        })
        print("FAILED:", row["filename"], e)

df_log = pd.DataFrame(download_log)
df_log.head()

Uploaded: nymeria/lightweight/20230607_s0_james_johnson_act3_ifj2gc/Nymeria_v0.0_20230607_s0_james_johnson_act3_ifj2gc_metadata.json
Uploaded: nymeria/lightweight/20230607_s0_james_johnson_act3_ifj2gc/Nymeria_v0.0_20230607_s0_james_johnson_act3_ifj2gc_narration_atomic_action.csv
Uploaded: nymeria/lightweight/20230607_s0_james_johnson_act3_ifj2gc/Nymeria_v0.0_20230607_s0_james_johnson_act3_ifj2gc_narration_activity_summarization.csv
Uploaded: nymeria/lightweight/20230607_s1_barbara_wheeler_act0_cvxtsi/Nymeria_v0.0_20230607_s1_barbara_wheeler_act0_cvxtsi_metadata.json
Uploaded: nymeria/lightweight/20230607_s1_barbara_wheeler_act0_cvxtsi/Nymeria_v0.0_20230607_s1_barbara_wheeler_act0_cvxtsi_narration_atomic_action.csv
Uploaded: nymeria/lightweight/20230607_s1_barbara_wheeler_act0_cvxtsi/Nymeria_v0.0_20230607_s1_barbara_wheeler_act0_cvxtsi_narration_activity_summarization.csv
Uploaded: nymeria/lightweight/20230607_s1_barbara_wheeler_act1_nkg6zo/Nymeria_v0.0_20230607_s1_barbara_wheeler_act1_

In [34]:
import boto3

s3 = boto3.client("s3")
BUCKET = "sagemaker-bst"
PREFIX = "nymeria/lightweight/"

count = 0
total_size = 0

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        count += 1
        total_size += obj["Size"]

print("Files uploaded:", count)
print("Size uploaded (MB):", total_size / (1024**2))

Files uploaded: 2809
Size uploaded (MB): 57.02054214477539


In [35]:
import io

csv_buffer = io.StringIO()
df_log.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=BUCKET,
    Key=f"{TARGET_PREFIX}download_log_all.csv",
    Body=csv_buffer.getvalue()
)

print("Saved full download log to S3")

Saved full download log to S3


## Load metadata JSONs from S3

In [36]:
import json

def read_json_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return json.loads(obj["Body"].read().decode("utf-8"))

metadata_keys = []

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET, Prefix=TARGET_PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("_metadata.json"):
            metadata_keys.append(key)

print("Metadata files found:", len(metadata_keys))
metadata_keys[:5]

Metadata files found: 1100


['nymeria/lightweight/20230607_s0_james_johnson_act0_e72nhq/Nymeria_v0.0_20230607_s0_james_johnson_act0_e72nhq_metadata.json',
 'nymeria/lightweight/20230607_s0_james_johnson_act1_7xwm28/Nymeria_v0.0_20230607_s0_james_johnson_act1_7xwm28_metadata.json',
 'nymeria/lightweight/20230607_s0_james_johnson_act2_yhbvpa/Nymeria_v0.0_20230607_s0_james_johnson_act2_yhbvpa_metadata.json',
 'nymeria/lightweight/20230607_s0_james_johnson_act3_ifj2gc/Nymeria_v0.0_20230607_s0_james_johnson_act3_ifj2gc_metadata.json',
 'nymeria/lightweight/20230607_s1_barbara_wheeler_act0_cvxtsi/Nymeria_v0.0_20230607_s1_barbara_wheeler_act0_cvxtsi_metadata.json']

In [37]:
metadata_rows = []

for key in metadata_keys:
    try:
        record = read_json_from_s3(BUCKET, key)
        record["s3_key"] = key
        metadata_rows.append(record)
    except Exception as e:
        print("FAILED:", key, e)

df_meta = pd.DataFrame(metadata_rows)

print(df_meta.shape)
df_meta.head(10)

(1100, 45)


,license,version,date,session_id,fake_name,act_id,uid,location,script,action_duration_sec,...,atomic_action,activity_summarization,participant_gender,participant_height_cm,participant_weight_kg,participant_bmi,participant_age_group,participant_ethnicity,participant_xsens_suit_size,s3_key
0,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s0,james_johnson,act0,e72nhq,Loc_04,S14-By_my_desk,824.135,...,True,True,Female,158.0,49.0,20.1,25-30,East Asian,M,nymeria/lightweight/20230607_s0_james_johnson_...
1,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s0,james_johnson,act1,7xwm28,Loc_04,S1-Relax_at_home,862.262,...,True,True,Female,158.0,49.0,20.1,25-30,East Asian,M,nymeria/lightweight/20230607_s0_james_johnson_...
2,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s0,james_johnson,act2,yhbvpa,Loc_04,S19-Fresh_air,859.662,...,True,True,Female,158.0,49.0,20.1,25-30,East Asian,M,nymeria/lightweight/20230607_s0_james_johnson_...
3,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s0,james_johnson,act3,ifj2gc,Loc_04,S12-Game_night,824.668,...,True,True,Female,158.0,49.0,20.1,25-30,East Asian,M,nymeria/lightweight/20230607_s0_james_johnson_...
4,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s1,barbara_wheeler,act0,cvxtsi,Loc_04,S14-By_my_desk,973.378,...,True,True,Male,195.0,96.0,25.2,18-24,African American,XXL,nymeria/lightweight/20230607_s1_barbara_wheele...
5,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s1,barbara_wheeler,act1,nkg6zo,Loc_04,S1-Relax_at_home,895.390,...,True,True,Male,195.0,96.0,25.2,18-24,African American,XXL,nymeria/lightweight/20230607_s1_barbara_wheele...
6,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s1,barbara_wheeler,act2,ig1oym,Loc_04,S16-Simon_says,925.719,...,True,True,Male,195.0,96.0,25.2,18-24,African American,XXL,nymeria/lightweight/20230607_s1_barbara_wheele...
7,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s1,barbara_wheeler,act3,qx6buy,Loc_04,S16-Simon_says,793.340,...,True,True,Male,195.0,96.0,25.2,18-24,African American,XXL,nymeria/lightweight/20230607_s1_barbara_wheele...
8,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230607,s1,barbara_wheeler,act4,5adv12,Loc_04,S1-Relax_at_home,790.274,...,True,True,Male,195.0,96.0,25.2,18-24,African American,XXL,nymeria/lightweight/20230607_s1_barbara_wheele...
9,CC BY-NC 4.0 (https://creativecommons.org/lice...,0.0,20230608,s0,shelby_arroyo,act0,3ciwl8,Loc_04,S14-By_my_desk,902.989,...,True,True,Female,153.0,45.0,19.2,46-50,Caucasian,XL,nymeria/lightweight/20230608_s0_shelby_arroyo_...


## Atomic narration CSVs

In [38]:
import pandas as pd
import io

def read_csv_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj["Body"].read()))

atomic_keys = []

for page in paginator.paginate(Bucket=BUCKET, Prefix=TARGET_PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("_narration_atomic_action.csv"):
            atomic_keys.append(key)

print("Atomic narration files:", len(atomic_keys))

Atomic narration files: 845


In [39]:
atomic_frames = []

for key in atomic_keys:
    try:
        tmp = read_csv_from_s3(BUCKET, key)
        tmp["s3_key"] = key
        atomic_frames.append(tmp)
    except Exception as e:
        print("FAILED:", key, e)

df_atomic = pd.concat(atomic_frames, ignore_index=True) if atomic_frames else pd.DataFrame()

print(df_atomic.shape)
df_atomic.head()

(173616, 8)


,request_id,gaia_id,start_time,end_time,annotator,creation_time,Describe my atomic actions,s3_key
0,4dad79c8-6d81-4069-9c8a-cce79a079ba2,1.161925e+15,5831.823751,5836.822947,7.019527e+15,1.720073e+09,C is standing in the foyer while talking to he...,nymeria/lightweight/20230607_s0_james_johnson_...
1,4dad79c8-6d81-4069-9c8a-cce79a079ba2,1.161925e+15,5836.822947,5841.822149,7.019527e+15,1.720073e+09,C is walking towards the hallway then turns he...,nymeria/lightweight/20230607_s0_james_johnson_...
2,4dad79c8-6d81-4069-9c8a-cce79a079ba2,1.161925e+15,5841.822149,5846.821344,7.019527e+15,1.720073e+09,C is standing in the hallway as she opens and ...,nymeria/lightweight/20230607_s0_james_johnson_...
3,4dad79c8-6d81-4069-9c8a-cce79a079ba2,1.161925e+15,5846.821344,5851.820544,7.019527e+15,1.720073e+09,C is standing in the hallway as she opens the ...,nymeria/lightweight/20230607_s0_james_johnson_...
4,4dad79c8-6d81-4069-9c8a-cce79a079ba2,1.161925e+15,5851.820544,5856.819743,7.019527e+15,1.720073e+09,"In the hallway, C slightly turns her body to t...",nymeria/lightweight/20230607_s0_james_johnson_...
